In [1]:
import json
from typing import List

# Unstructured for document parsing
from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title

# LangChain components (not used in this notebook)
# from dotenv import load_dotenv

# load_dotenv()

c:\Program Files\Python39\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 1) read PDF

In [2]:
def partition_document(file_path: str):
    """Extract elements from PDF using unstructured"""
    print(f"📄 Partitioning document: {file_path}")
    
    elements = partition_pdf(
        filename=file_path,  # Path to your PDF file
        strategy="hi_res", # Use the most accurate (but slower) processing method of extraction
        infer_table_structure=True, # Keep tables as structured HTML, not jumbled text
        extract_image_block_types=["Image"], # Grab images found in the PDF
        extract_image_block_to_payload=True # Store images as base64 data you can actually use
    )
    
    print(f"✅ Extracted {len(elements)} elements")
    return elements

# Test with your PDF file
file_path = "environment.pdf"  # Change this to your PDF path
elements = partition_document(file_path)

📄 Partitioning document: environment.pdf


The `max_size` parameter is deprecated and will be removed in v4.26. Please specify in `size['longest_edge'] instead`.


✅ Extracted 4956 elements


In [3]:
set([str(type(el)) for el in elements])

{"<class 'unstructured.documents.elements.FigureCaption'>",
 "<class 'unstructured.documents.elements.Header'>",
 "<class 'unstructured.documents.elements.Image'>",
 "<class 'unstructured.documents.elements.ListItem'>",
 "<class 'unstructured.documents.elements.NarrativeText'>",
 "<class 'unstructured.documents.elements.Table'>",
 "<class 'unstructured.documents.elements.Text'>",
 "<class 'unstructured.documents.elements.Title'>"}

In [4]:
theme_list = ["<class 'unstructured.documents.elements.FigureCaption'>",
 "<class 'unstructured.documents.elements.Header'>",
 "<class 'unstructured.documents.elements.Image'>",
 "<class 'unstructured.documents.elements.ListItem'>",
 "<class 'unstructured.documents.elements.NarrativeText'>",
 "<class 'unstructured.documents.elements.Table'>",
 "<class 'unstructured.documents.elements.Text'>",
 "<class 'unstructured.documents.elements.Title'>"]



In [5]:
from collections import Counter

# Count occurrences of each type in theme_list within elements
type_counts = Counter(str(type(el)) for el in elements if str(type(el)) in theme_list)

# Display the counts
for theme in theme_list:
    count = type_counts.get(theme, 0)
    print(f"{theme}: {count}")

<class 'unstructured.documents.elements.FigureCaption'>: 10
<class 'unstructured.documents.elements.Header'>: 265
<class 'unstructured.documents.elements.Image'>: 2244
<class 'unstructured.documents.elements.ListItem'>: 1078
<class 'unstructured.documents.elements.NarrativeText'>: 272
<class 'unstructured.documents.elements.Table'>: 65
<class 'unstructured.documents.elements.Text'>: 536
<class 'unstructured.documents.elements.Title'>: 486


In [6]:
count = 0
for i in range(len(elements)):
    element = elements[i]
    if str(type(element)) == "<class 'unstructured.documents.elements.Title'>" :
        base_dict = element.to_dict()
        print(f"Element {i} is a Title with text: {base_dict['text']}")
        count += 1
        # print(base_dict)
        # break
print(count)

Element 9 is a Title with text: ENVIRONMENT
Element 10 is a Title with text: Table of Contents
Element 100 is a Title with text: Copyright © by Vision IAS
Element 109 is a Title with text: 1. BIODIVERSITY
Element 110 is a Title with text: 1.1. WILDLIFE AND CONSERVATION
Element 111 is a Title with text: 1.1.1. IUCN WORLD CONSERVATION CONGRESS
Element 112 is a Title with text: Why in the News?
Element 114 is a Title with text: Key Resolutions at Member’s Assembly
Element 115 is a Title with text: • Adoption of a ‘Unite for Nature on the Path to 2045’- 20-year Strategic Vision.
Element 122 is a Title with text: Related News: India’s National Red List Roadmap and Vision 2025–2030
Element 346 is a Title with text: 1.1.1.1. KEY INSTRUMENTS OF IUCN IN NEWS
Element 806 is a Title with text: Why in the news?
Element 827 is a Title with text: Related News: Barda Wildlife Sanctuary
Element 832 is a Title with text: 1.1.2. UNESCO’S WORLD NETWORK OF BIOSPHERE RESERVES (WNBR)
Element 833 is a Title 

# 2) Read and collect image chunks

In [7]:
import base64
from PIL import Image
import io
import pytesseract

image_chunks = []
for i in range(len(elements)):
    element = elements[i]
    if str(type(element)) == "<class 'unstructured.documents.elements.Image'>":
         base64_string = element.to_dict()['metadata']['image_base64']
         img_data = base64.b64decode(base64_string)
         img = Image.open(io.BytesIO(img_data))
         text = pytesseract.image_to_string(img).strip()
         if len(text) > 10:
                print(f"Element {i} is an Image with extracted text: {text}")
                image_chunks.append(text)

Element 0 is an Image with extracted text: =
‘~~ INSPIRING INNOVATION

Years of Excellence

~ Classroom Study Material —

(April 2025 to November 2025)

AHMEDABAD | BENGALURU | BHOPAL | CHANDIGARH | DELHI | GUWAHATI | HYDERABAD | JAIPUR | JODHPUR | LUCKNOW | PRAYAGRAJ | PUNE | RANCHI 8468022022

@&) enquiry@visionias.in © /ewisiontasdethi (@) /visionias_upsc ©) vision_ias /Nision|AS_UPSC (©) wowwisionias.in 9019066066
Element 2 is an Image with extracted text: ~

Classroom Study Material
(April 2025 to November 2025)

7

i r
() 84680:
D | BENGALURU | BHOPAL | CHANDIGARH | DELHI | GUWAHATI | HYDERABAD | JAIPUR | JODHPUR| LUCKNOW | PRAYAGRAJ|PUNE|RANCHI \SZ

iry@visionias.in © /enisiontasdethi (@) /visionias_upsc © vision_ias /Nision|AS_UPSC (©) wwwwisionias.in 90190
Element 5 is an Image with extracted text: i, ee
8468022022

9019066066
Element 93 is an Image with extracted text: o) wy. s Scan to take the PT 365 Smart
| Pilla Quiz and test your knowledge on

our online platform!
Element

# 3) Filter + chunk OCR text with Groq (Environment only)

In [53]:
import os
import re
import json
import time
from typing import Dict, List

from dotenv import load_dotenv
from groq import Groq

load_dotenv()

GROQ_API_KEY = os.getenv('GROQ_API_KEY')
if not GROQ_API_KEY:
    raise ValueError('Missing GROQ_API_KEY in .env')

client = Groq(api_key=GROQ_API_KEY)

# Pick a fast, free-tier friendly model.
GROQ_MODEL = 'llama-3.1-8b-instant'


In [54]:
def clean_text(text: str) -> str:
    text = text.replace('\u00a0', ' ')
    text = re.sub(r'\s+', ' ', text).strip()
    return text


AD_HINTS = [
    'vision ias', 'foundation course', 'prelims', 'mains', 'telegram', 'scan qr',
    'call', 'whatsapp', 'admission', 'batch', 'fee', 'contact', 'register',
    'venue', 'timing', 'limited seats', 'app download', 'join now'
]

ENV_KEYWORDS = [
    'biodiversity', 'ecosystem', 'species', 'habitat', 'wetland', 'forest',
    'climate', 'pollution', 'conservation', 'protected area', 'national park',
    'wildlife', 'endangered', 'carbon', 'emission', 'sustainability', 'treaty',
    'act', 'policy', 'reserve', 'sanctuary', 'river', 'ocean', 'mangrove',
    'coral', 'glacier', 'biodiversity', 'renewable', 'energy', 'IUCN', 
    "Disaster Affected Areas", "wildlife", "deforestation", "reforestation", 
    "climate change", "global warming", "sustainability", "carbon footprint", 
    "greenhouse gases", "renewable energy", "biodiversity loss", 
    "conservation efforts", "environmental policy", "ecosystem services", 
    "climate action", "sustainable development", "environmental impact", 
    "natural resources", "environmental education", "green technology", 
    "environmental justice", "environmental conservation", "environmental awareness", 
    "environmental sustainability", "environmental protection", 
    "biosphere", "deforestation", "reforestation", "climate", "warming",
    "sustainability", "ocean", "river", "forest", "wildlife", "biodiversity", "pollution",
    "efficiency", "energy", "solar", "ocean"
]


def has_env_signal(text: str) -> bool:
    t = text.lower()
    return any(k in t for k in ENV_KEYWORDS)


def is_low_signal(text: str) -> bool:
    t = text.lower()
    # If there is strong environment signal, do not drop early
    if has_env_signal(t):
        return False
    if len(t) < 25:
        return True
    # Heavy contact/marketing pattern (more lenient)
    if sum(ch.isdigit() for ch in t) > max(16, len(t) * 0.40):
        return True
    if any(h in t for h in AD_HINTS):
        return True
    return False


In [59]:
# SYSTEM_PROMPT = (
#     'You are a careful filter for UPSC study notes. '
#     'Decide if the input text is primarily about Environment or related topics and facts '
#     '(ecology, biodiversity, climate change, pollution, conservation, environmental laws, '
#     'treaties, institutions, geography/ecosystems, environmental disasters, sustainability). '
#     'Include concise factual items, species names, protected areas, reports/indices, '
#     'and short definitions if they are environmental. '
#     'Reject only if the text is clearly an ad/promo, coaching info, schedules, contact info, '
#     'or unrelated (politics, economy, sports, entertainment, etc.). '
#     'If uncertain, lean towards keeping the text and convert to relevant information that will help in UPSC preparation. '
#     'If relevant, keep the key facts with minimal loss in a concise chunk. '
#     'Add a one-word heading that best labels the chunk. '
#     'Make the chunk gramatically correct and self-contained. '
#     'Return JSON only: {keep: true|false, reason: ..., heading: ..., chunk: ...}. '
#     'If keep=false, heading and chunk must be empty strings.'
# )

SYSTEM_PROMPT = (
    'You are a careful filter for UPSC study notes. '
    'Decide if the input text is primarily about UPSC or related topics and facts '
    'Include concise factual items, species names, protected areas, reports/indices, '
    'and short definitions. '
    'Reject only if the text is clearly an ad/promo, coaching info, schedules, contact info, '
    'or unrelated (politics, economy, sports, entertainment, etc.). '
    'If uncertain, lean towards keeping the text and convert to relevant information that will help in UPSC preparation. '
    'If relevant, keep the key facts with minimal loss in a concise chunk. '
    'Make the chunk gramatically correct and self-contained. '
    'Return JSON only: {keep: true|false, reason: ..., heading: ..., chunk: ...}. '
    'If keep=false, heading and chunk must be empty strings.'
)


def groq_filter_chunk(text: str) -> Dict[str, str]:
    user_prompt = f'Input text:\n{text}'
    try:
        resp = client.chat.completions.create(
            model=GROQ_MODEL,
            temperature=0.2,
            max_tokens=380,
            messages=[
                {'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user', 'content': user_prompt},
            ],
            response_format={'type': 'json_object'},
        )
    except Exception:
        # Fallback if JSON mode is unsupported on the current model/account
        resp = client.chat.completions.create(
            model=GROQ_MODEL,
            temperature=0.2,
            max_tokens=380,
            messages=[
                {'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user', 'content': user_prompt + '\n\nReturn only JSON.'},
            ],
        )
    content = resp.choices[0].message.content
    try:
        data = json.loads(content)
    except json.JSONDecodeError:
        # Fallback: attempt to extract JSON-like object
        start = content.find('{')
        end = content.rfind('}')
        if start != -1 and end != -1 and end > start:
            data = json.loads(content[start:end+1])
        else:
            data = {'keep': False, 'reason': 'parse_error', 'heading': '', 'chunk': ''}
    # Normalize
    data['keep'] = bool(data.get('keep', False))
    data['reason'] = str(data.get('reason', '')).strip()
    data['heading'] = str(data.get('heading', '')).strip()
    data['chunk'] = str(data.get('chunk', '')).strip()
    return data


In [60]:
# Use your OCR list here. If you already built image_chunks, this will use it.
texts = image_chunks if 'image_chunks' in globals() else []

relevant_chunks: List[Dict[str, str]] = []
discarded: List[Dict[str, str]] = []

def normalize_heading(h: str) -> str:
    h = re.sub(r"[^A-Za-z]", "", h).strip()
    if not h:
        return "Environment"
    return h[:20]

for idx, raw in enumerate(texts):
    text = clean_text(raw)
    if is_low_signal(text):
        discarded.append({'index': idx, 'reason': 'low_signal', 'text': text})
        continue

    try:
        result = groq_filter_chunk(text)
    except Exception as e:
        discarded.append({'index': idx, 'reason': f'api_error: {e}', 'text': text})
        continue

    if result.get('keep') and result.get('chunk'):
        heading = normalize_heading(result.get('heading', ''))
        chunk = '{}: {}'.format(heading, result['chunk'])
        relevant_chunks.append({
            'index': idx,
            'heading': heading,
            'chunk': chunk,
            'reason': result.get('reason', ''),
            'source_text': text,
        })
    else:
        discarded.append({
            'index': idx,
            'reason': result.get('reason', 'not_relevant'),
            'text': text,
        })

    # Gentle rate limiting for free tier
    time.sleep(0.15)

print(f'Relevant chunks: {len(relevant_chunks)}')
print(f'Discarded: {len(discarded)}')


Relevant chunks: 75
Discarded: 38


In [61]:
relevant_chunks

[{'index': 5,
  'heading': 'EnhancementsinStudyM',
  'chunk': "EnhancementsinStudyM: The study materials now include summarised infographics on key concepts like India's Nationally Determined Contributions, Biofuels, and Article 6 of the Paris Agreement. Important species and protected areas are also covered, along with pictorial and interactive thumbnails of protection status and recognition by CAI/TS, UNESCO's Man and Biosphere Programme, etc.",
  'reason': 'Related to UPSC study materials and features',
  'source_text': 'Note to Students Dear Students, [auiz PT 365 documents comprehensively cover the important current affairs of last 1 year (365 days) in a consolidated manner to aid Prelims preparation. In our endeavour to further enhance the document in the interest of the aspirants, following additions have been incorporated: Summarised Infographics: Topics such as: » Key information of major species. jp Key concepts like India’s Nationally Determined Contributions, Biofuels, Arti

In [62]:
discarded

[{'index': 0,
  'reason': 'Clearly an ad/promo',
  'text': '= ‘~~ INSPIRING INNOVATION Years of Excellence ~ Classroom Study Material — (April 2025 to November 2025) AHMEDABAD | BENGALURU | BHOPAL | CHANDIGARH | DELHI | GUWAHATI | HYDERABAD | JAIPUR | JODHPUR | LUCKNOW | PRAYAGRAJ | PUNE | RANCHI 8468022022 @&) enquiry@visionias.in © /ewisiontasdethi (@) /visionias_upsc ©) vision_ias /Nision|AS_UPSC (©) wowwisionias.in 9019066066'},
 {'index': 1,
  'reason': 'clearly an ad/promo',
  'text': '~ Classroom Study Material (April 2025 to November 2025) 7 i r () 84680: D | BENGALURU | BHOPAL | CHANDIGARH | DELHI | GUWAHATI | HYDERABAD | JAIPUR | JODHPUR| LUCKNOW | PRAYAGRAJ|PUNE|RANCHI \\SZ iry@visionias.in © /enisiontasdethi (@) /visionias_upsc © vision_ias /Nision|AS_UPSC (©) wwwwisionias.in 90190'},
 {'index': 2, 'reason': 'low_signal', 'text': 'i, ee 8468022022 9019066066'},
 {'index': 3,
  'reason': 'clearly an ad/promo',
  'text': 'o) wy. s Scan to take the PT 365 Smart | Pilla Quiz an

In [22]:
# Save results
from pathlib import Path

out_dir = Path('data')
out_dir.mkdir(exist_ok=True)

jsonl_path = out_dir / 'environment_image_chunks.jsonl'
with jsonl_path.open('w', encoding='utf-8') as f:
    for item in relevant_chunks:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

text_path = out_dir / 'environment_image_chunks.txt'
with text_path.open('w', encoding='utf-8') as f:
    for item in relevant_chunks:
        f.write(item["chunk"] + "\n\n")

print(f'Wrote {len(relevant_chunks)} chunks to {jsonl_path} and {text_path}')


Wrote 93 chunks to data\environment_image_chunks.jsonl and data\environment_image_chunks.txt


# 4) Combine text and others and chunk page wise

In [25]:
from collections import defaultdict
from pathlib import Path
from typing import Dict, Any

# 4.1) Build page chunks from selected element types
allowed_types = {
    "<class 'unstructured.documents.elements.ListItem'>",
    "<class 'unstructured.documents.elements.NarrativeText'>",
    "<class 'unstructured.documents.elements.Table'>",
    "<class 'unstructured.documents.elements.Text'>",
    "<class 'unstructured.documents.elements.Title'>",
}

def _element_text(el) -> str:
    text = getattr(el, "text", "") or ""
    if not text and str(type(el)) == "<class 'unstructured.documents.elements.Table'>":
        text = getattr(el.metadata, "text_as_html", "") or ""
    return text.strip()

page_buckets: Dict[int, list] = defaultdict(list)
for el in elements:
    if str(type(el)) not in allowed_types:
        continue
    page_num = getattr(el.metadata, "page_number", None)
    if page_num is None:
        continue
    text = _element_text(el)
    if text:
        page_buckets[int(page_num)].append(text)

page_chunks = []
for page_num in sorted(page_buckets.keys()):
    page_text = "\n".join(page_buckets[page_num]).strip()
    if page_text:
        page_chunks.append({"page_number": page_num, "text": page_text})

print(f"Built {len(page_chunks)} page chunks")

Built 136 page chunks


In [29]:
page_chunks = page_chunks[5:-2]

In [30]:
# 4.2) LLM-based micro-chunking per page (Groq)

LLM_PROMPT = (
    "You will receive text from a single PDF page of UPSC Environment study notes. "
    "Split into small, self-contained information chunks (short paragraphs). "
    "For each kept chunk, improve clarity and grammar while preserving facts and key terms. "
    "Do not add new information. Preserve acts, dates, species names, and place names. "
    "If text looks like ads/promos/coaching/telegram/QR/fee/contact info, discard it. "
    "If a line mixes ads with study content, keep only the study content. "
    "Return JSON only in the form: {\"chunks\": [{\"index\": 1, \"heading\": \"...\", \"chunk\": \"...\", \"reason\": \"...\", \"source_text\": \"...\"}, ...]}. "
    "Use 'reason' to explain keep/discard. For discarded items, set heading and chunk to empty strings but include source_text."
)

def chunk_page_with_llm(page_text: str) -> list:
    max_chars = 6000
    if len(page_text) <= max_chars:
        segments = [page_text]
    else:
        paras = [p for p in page_text.split("\n") if p.strip()]
        segments = []
        current = ""
        for p in paras:
            if current and (len(current) + len(p) + 1 > max_chars):
                segments.append(current)
                current = p
            else:
                current = (current + "\n" + p).strip() if current else p
        if current:
            segments.append(current)

    all_chunks = []
    for seg in segments:
        resp = client.chat.completions.create(
            model=GROQ_MODEL,
            messages=[{"role": "user", "content": f"{LLM_PROMPT}\n\nTEXT:\n{seg}"}],
            temperature=0,
        )
        raw = (resp.choices[0].message.content or "").strip()
        try:
            data = json.loads(raw)
            chunks = data.get("chunks", [])
            for c in chunks:
                if isinstance(c, dict):
                    all_chunks.append(c)
        except Exception:
            # Fallback: keep each non-empty line as a minimal chunk
            idx = 1
            for line in seg.split("\n"):
                if line.strip():
                    all_chunks.append({
                        "index": idx,
                        "heading": "",
                        "chunk": line.strip(),
                        "reason": "fallback_no_llm",
                        "source_text": line.strip(),
                    })
                    idx += 1

    return [c for c in all_chunks if isinstance(c, dict)]

llm_chunks = []
for page in page_chunks:
    pieces = chunk_page_with_llm(page["text"])
    for i, piece in enumerate(pieces, start=1):
        llm_chunks.append({
            "page_number": page["page_number"],
            "chunk_index": int(piece.get("index", i)),
            "heading": str(piece.get("heading", "")).strip(),
            "chunk": str(piece.get("chunk", "")).strip(),
            "reason": str(piece.get("reason", "")).strip(),
            "source_text": str(piece.get("source_text", "")).strip(),
        })

print(f"Built {len(llm_chunks)} LLM chunks from {len(page_chunks)} pages")

Built 1595 LLM chunks from 129 pages


In [32]:
# Standalone merge-by-heading (consecutive) using llm_chunks
def merge_consecutive_by_heading(llm_chunks):
    # Sort to ensure original order
    items = sorted(llm_chunks, key=lambda x: (x.get("page_number", 0), x.get("chunk_index", 0)))

    merged = []
    current = None

    def norm_heading(h):
        return (h or "").strip().lower()

    for item in items:
        heading = item.get("heading", "")
        chunk = item.get("chunk", "")
        source_text = item.get("source_text", "")
        reason = item.get("reason", "")

        if current is None:
            current = {
                "heading": heading,
                "chunk": chunk,
                "source_text": source_text,
                "reason": reason,
                "page_numbers": [item.get("page_number")],
                "chunk_indices": [item.get("chunk_index")],
            }
            continue

        if norm_heading(heading) == norm_heading(current["heading"]):
            # Merge into current
            if chunk:
                current["chunk"] = (current["chunk"] + "\n" + chunk).strip()
            if source_text:
                current["source_text"] = (current["source_text"] + "\n" + source_text).strip()
            if reason and reason not in current["reason"]:
                current["reason"] = (current["reason"] + " | " + reason).strip(" |")
            current["page_numbers"].append(item.get("page_number"))
            current["chunk_indices"].append(item.get("chunk_index"))
        else:
            merged.append(current)
            current = {
                "heading": heading,
                "chunk": chunk,
                "source_text": source_text,
                "reason": reason,
                "page_numbers": [item.get("page_number")],
                "chunk_indices": [item.get("chunk_index")],
            }

    if current:
        merged.append(current)

    return merged

merged_chunks = merge_consecutive_by_heading(llm_chunks)
print(f"Merged {len(llm_chunks)} -> {len(merged_chunks)}")


Merged 1595 -> 646


In [33]:
merged_chunks

[{'heading': 'IUCN World Conservation Congress',
  'chunk': 'International Union for Conservation of Nature (IUCN) World Conservation Congress 2025 took place in Abu Dhabi, UAE.',
  'source_text': 'International Union for Conservation of Nature (IUCN) World Conservation Congress 2025 took place in Abu Dhabi, UAE.',
  'reason': 'Study content',
  'page_numbers': [6],
  'chunk_indices': [1]},
 {'heading': 'Key Resolutions at Member’s Assembly',
  'chunk': 'Adoption of a ‘Unite for Nature on the Path to 2045’- 20-year Strategic Vision.\nAbu Dhabi Call to Action to accelerate action across 5 key areas – reaffirming nature as foundation of well-being, strengthening multilateralism, ensuring justice and inclusion, advancing knowledge and innovation, and scaling up resources for nature and climate action.\nOver 100 new members including six states – Armenia, Tajikistan, Marshall Islands, Gabon, Tuvalu, and Zimbabwe.\nMotion advocating for a just phase-out of fossil fuels adopted for the first

TypeError: unhashable type: 'list'

In [72]:
keys_to_keep = ["heading", "chunk", "reason", "source_text"]
text_chunks=[]
for i, chunk in enumerate(merged_chunks):
    new_dict = {k: chunk[k] for k in keys_to_keep if k in chunk}
    new_dict["index"]=i
    text_chunks.append(new_dict)

In [73]:
text_chunks

[{'heading': 'IUCN World Conservation Congress',
  'chunk': 'International Union for Conservation of Nature (IUCN) World Conservation Congress 2025 took place in Abu Dhabi, UAE.',
  'reason': 'Study content',
  'source_text': 'International Union for Conservation of Nature (IUCN) World Conservation Congress 2025 took place in Abu Dhabi, UAE.',
  'index': 0},
 {'heading': 'Key Resolutions at Member’s Assembly',
  'chunk': 'Adoption of a ‘Unite for Nature on the Path to 2045’- 20-year Strategic Vision.\nAbu Dhabi Call to Action to accelerate action across 5 key areas – reaffirming nature as foundation of well-being, strengthening multilateralism, ensuring justice and inclusion, advancing knowledge and innovation, and scaling up resources for nature and climate action.\nOver 100 new members including six states – Armenia, Tajikistan, Marshall Islands, Gabon, Tuvalu, and Zimbabwe.\nMotion advocating for a just phase-out of fossil fuels adopted for the first time.\nWild animals recognised a

In [74]:
# Combine image chunks and append to a new JSONL
import json
from pathlib import Path

in_path = Path("data/environment_image_chunks.jsonl")
out_path = Path("data/environment_chunks.jsonl")

image_chunks = []
with in_path.open("r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        image_chunks.append(json.loads(line))


In [80]:
combined_chunks = image_chunks + text_chunks

In [81]:
out_dir = Path('data')
out_dir.mkdir(exist_ok=True)

jsonl_path = out_dir / 'environment_chunks.jsonl'
with jsonl_path.open('w', encoding='utf-8') as f:
    for item in combined_chunks:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")
print(f"Wrote {len(combined_chunks)} combined chunks to {jsonl_path}")

Wrote 739 combined chunks to data\environment_chunks.jsonl


In [1]:
from docling.document_converter import DocumentConverter

source = "environment.pdf"
converter = DocumentConverter()
doc = converter.convert(source).document
print(doc.export_to_markdown())

c:\Program Files\Python39\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[INFO] 2026-03-20 00:38:53,042 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-20 00:38:53,066 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\91963\AppData\Roaming\Python\Python39\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-03-20 00:38:53,070 [RapidOCR] main.py:53: Using C:\Users\91963\AppData\Roaming\Python\Python39\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-03-20 00:38:53,285 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-20 00:38:53,288 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\91963\AppData\Roaming\Python\Python39\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-03-20 00:3

<!-- image -->

## ENVIRONMENT

Classroom Study Material

(April 2025 to November 2025)

AHMEDABAD|BENGALURU|BHOPAL|CHANDIGARH|DELHI|GUWAHATI|HYDERABAD|JAIPUR|JODHPUR|LUCKNOW|PRAYAGRAJ|PUNE|RANCHI

<!-- image -->

<!-- image -->

<!-- image -->

<!-- image -->

<!-- image -->

<!-- image -->

<!-- image -->

<!-- image -->

<!-- image -->

## ENVIRONMENT

## Table of Contents

| 1. BIODIVERSITY______________________                                                                                           | 1. BIODIVERSITY______________________                                                 |
|---------------------------------------------------------------------------------------------------------------------------------|---------------------------------------------------------------------------------------|
| 1.1. Wildlife and Conservation ___________ 5                                                                                    | 1.1. Wildlife and Conservation ___________ 5    

In [ ]:
XCGVprint(doc.export_to_markdown())

NameError: name 'doc' is not defined

In [2]:
doc

NameError: name 'doc' is not defined

In [6]:
# Docling + OCR image-text insertion (creates a new .md file)
# If needed: pip install pdfplumber pdf2image rapidocr-onnxruntime pillow numpy
from docling.document_converter import DocumentConverter
from pathlib import Path
import re
import pdfplumber
from pdf2image import convert_from_path
import numpy as np
from rapidocr_onnxruntime import RapidOCR

def docling_to_markdown_with_ocr(pdf_path, out_md_path):
    pdf_path = Path(pdf_path)
    if out_md_path is None:
        out_md_path = pdf_path.with_suffix('').name + '_with_ocr.md'
    out_md_path = Path(out_md_path)

    # Docling conversion
    converter = DocumentConverter()
    doc = converter.convert(str(pdf_path)).document
    md = doc.export_to_markdown()

    # OCR setup
    ocr = RapidOCR()
    ocr_texts = []

    with pdfplumber.open(str(pdf_path)) as pdf:
        pages = convert_from_path(str(pdf_path), dpi=300)
        for page, pil_img in zip(pdf.pages, pages):
            images = page.images
            if not images:
                continue
            page_img = np.array(pil_img)
            page_width_pt = page.width
            page_height_pt = page.height
            img_h, img_w = page_img.shape[:2]
            sx = img_w / page_width_pt
            sy = img_h / page_height_pt

            images_sorted = sorted(images, key=lambda im: (im['top'], im['x0']))
            for im in images_sorted:
                x0 = int(im['x0'] * sx)
                x1 = int(im['x1'] * sx)
                top = int(im['top'] * sy)
                bottom = int(im['bottom'] * sy)
                crop = page_img[top:bottom, x0:x1]
                result, _ = ocr(crop)
                text = ' '.join([r[1] for r in result]) if result else ''
                ocr_texts.append(text.strip())

    placeholders = len(re.findall(r'<!-- image -->', md))
    if placeholders != len(ocr_texts):
        print(f'Warning: placeholders={placeholders}, ocr_texts={len(ocr_texts)}')

    for txt in ocr_texts:
        replacement = txt if txt else '[no OCR text]'
        md = md.replace('<!-- image -->', replacement, 1)

    out_md_path.write_text(md, encoding='utf-8')
    print(f'Saved: {out_md_path}')
    return out_md_path

# ---- Usage ----
docling_to_markdown_with_ocr('environment.pdf', 'environment_with_ocr.md')


c:\Program Files\Python39\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-03-20 00:27:39,717 - RapidOCR - INFO: Using engine_name: onnxruntime
2026-03-20 00:27:39,769 - RapidOCR - INFO: File exists and is valid: C:\Users\91963\AppData\Roaming\Python\Python39\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.onnx
2026-03-20 00:27:39,769 - RapidOCR - INFO: Using C:\Users\91963\AppData\Roaming\Python\Python39\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.onnx
2026-03-20 00:27:39,995 - RapidOCR - INFO: Using engine_name: onnxruntime
2026-03-20 00:27:40,021 - RapidOCR - INFO: File exists and is valid: C:\Users\91963\AppData\Roaming\Python\Python39\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_infer.onnx
2026-03-20 00:27:40,025 - RapidOCR - INFO: Using C:\Users\91963\AppData\Roaming\Python\Python39

: 

Saved: environment.md
